# Kraken Fine-tuning — CREMMA Médiéval

Fine-tuning du modèle `cremma-medieval_best` sur les manuscrits médiévaux français.

**⚠️ IMPORTANT:** Sauvegarder la version (Save Version) dès que le premier epoch finit!

In [ ]:
# CELL 1 — Install
%%capture
!pip install numpy==1.26.4
!pip install scipy==1.13.1 scikit-learn==1.5.2
!pip install kraken==5.3.0
!pip install editdistance Pillow>=10.0 lxml

In [ ]:
# CELL 2 — Setup + download model
import os
os.environ['PATH'] += ':/root/.local/bin:/usr/local/bin:/opt/conda/bin'
os.makedirs('/kaggle/working/models', exist_ok=True)

# Download pre-trained CREMMA model
!kraken get 10.5281/zenodo.7234166

import glob
model_files = glob.glob('/root/.config/kraken/**/*.mlmodel', recursive=True)
PRETRAINED = model_files[0] if model_files else None
print(f'Pre-trained model: {PRETRAINED}')

In [ ]:
# CELL 3 — Data prep (5 min)
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

!git clone https://github.com/HTR-United/cremma-medieval data/cremma 2>/dev/null || echo 'Already cloned'

lines_dir = Path('data/lines')
lines_dir.mkdir(parents=True, exist_ok=True)
all_records = []
cremma_dir = Path('data/cremma/data')
line_counter = 0

for ms_dir in sorted(cremma_dir.iterdir()):
    if not ms_dir.is_dir():
        continue
    ms_name = ms_dir.name
    for xml_path in sorted(f for f in ms_dir.glob('*.xml') if 'chocomufin' not in f.name):
        img_path = xml_path.with_suffix('.jpg')
        if not img_path.exists():
            continue
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
        except:
            continue
        ns = root.tag.split('}')[0] + '}' if root.tag.startswith('{') else ''
        try:
            page_img = Image.open(img_path).convert('RGB')
        except:
            continue
        for i, text_line in enumerate(root.iter(f'{ns}TextLine')):
            parts = [s.get('CONTENT', '') for s in text_line.iter(f'{ns}String') if s.get('CONTENT')]
            text = ' '.join(parts).strip()
            if len(text) < 2:
                continue
            try:
                hpos, vpos = float(text_line.get('HPOS', 0)), float(text_line.get('VPOS', 0))
                width, height = float(text_line.get('WIDTH', 0)), float(text_line.get('HEIGHT', 0))
            except:
                continue
            if width <= 0 or height <= 0:
                continue
            x0, y0 = max(0, int(hpos)), max(0, int(vpos))
            x1, y1 = min(page_img.width, int(hpos+width)), min(page_img.height, int(vpos+height))
            if x1 <= x0 or y1 <= y0:
                continue
            line_img_path = lines_dir / f'{line_counter:05d}.png'
            page_img.crop((x0, y0, x1, y1)).save(line_img_path)
            all_records.append({'img_path': str(line_img_path), 'text': text, 'manuscript': ms_name})
            line_counter += 1

print(f'Extracted: {len(all_records)} lines')

# Split
manuscripts_list = [r['manuscript'] for r in all_records]
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(all_records, groups=manuscripts_list))
train_records = [all_records[i] for i in train_idx]
val_records = [all_records[i] for i in val_idx]
print(f'Train: {len(train_records)}, Val: {len(val_records)}')

# Create manifests
kraken_train_dir = Path('data/kraken_train')
kraken_val_dir = Path('data/kraken_val')
kraken_train_dir.mkdir(parents=True, exist_ok=True)
kraken_val_dir.mkdir(parents=True, exist_ok=True)

train_manifest, val_manifest = [], []
for i, r in enumerate(train_records):
    p = kraken_train_dir / f'line_{i:05d}.png'
    Image.open(r['img_path']).convert('L').save(p)
    (kraken_train_dir / f'line_{i:05d}.gt.txt').write_text(r['text'], encoding='utf-8')
    train_manifest.append(str(p))
for i, r in enumerate(val_records):
    p = kraken_val_dir / f'line_{i:05d}.png'
    Image.open(r['img_path']).convert('L').save(p)
    (kraken_val_dir / f'line_{i:05d}.gt.txt').write_text(r['text'], encoding='utf-8')
    val_manifest.append(str(p))

Path('train_manifest.txt').write_text('\n'.join(train_manifest))
Path('val_manifest.txt').write_text('\n'.join(val_manifest))
print(f'Manifests ready: {len(train_manifest)} train, {len(val_manifest)} val')

In [ ]:
# CELL 4 — TRAIN (save version after first epoch!)
%%time

!ketos train \
    -f path \
    -d cuda:0 \
    -i {PRETRAINED} \
    --resize add \
    --augment \
    --workers 4 \
    --batch-size 8 \
    --lag 5 \
    --min-epochs 3 \
    --epochs 30 \
    -o /kaggle/working/models/kraken_final \
    --training-files train_manifest.txt \
    --evaluation-files val_manifest.txt

In [ ]:
# CELL 5 — Check results
import glob
models = sorted(glob.glob('/kaggle/working/models/kraken_final*.mlmodel'))
print(f'Models saved: {models}')
if models:
    print(f'Best model: {models[-1]}')

In [ ]:
# CELL 6 — Evaluate best model
import warnings
warnings.filterwarnings('ignore')
import editdistance
from kraken import rpred
from kraken.lib import models as kraken_models
from kraken.containers import BBoxLine, Segmentation
from tqdm.notebook import tqdm

def compute_cer(predictions, references):
    total_errors = sum(editdistance.eval(p, r) for p, r in zip(predictions, references))
    total_chars = sum(len(r) for r in references)
    return total_errors / total_chars if total_chars > 0 else 0.0

# Load best model
best_model_path = sorted(glob.glob('/kaggle/working/models/kraken_final*.mlmodel'))[-1]
model = kraken_models.load_any(best_model_path)
print(f'Evaluating: {best_model_path}')

preds, refs = [], []
for record in tqdm(val_records[:100], desc='Evaluating'):
    img = Image.open(record['img_path']).convert('L')
    w, h = img.size
    try:
        line = BBoxLine(id='l0', bbox=(0, 0, w, h), text_direction='horizontal-lr')
        bounds = Segmentation(lines=[line], imagename='l', type='bbox',
                              text_direction='horizontal-lr', script_detection=False,
                              line_orders=[], regions={})
        pred = rpred.rpred(model, img, bounds)
        text = ''.join([r.prediction for r in pred])
    except:
        text = ''
    preds.append(text)
    refs.append(record['text'])

cer = compute_cer(preds, refs)
print(f'\n{"="*50}')
print(f'Kraken Fine-tuned CER: {cer:.1%}')
print(f'Kraken Zero-shot CER:  32.6%')
print(f'Improvement: {32.6 - cer*100:.1f} points')
print(f'{"="*50}')